# SomnoGuard: analítica, visión y ML empresarial
Notebook académico reproducible. Usa datos sintéticos marcados como demostración o un CSV anonimizado exportado por la web. **No cargues credenciales, nombres, correos ni videos sin consentimiento.**

In [ ]:
!pip -q install numpy pandas scikit-learn tensorflow keras opencv-python-headless matplotlib seaborn
from pathlib import Path
import json, sys
ROOT = Path.cwd()
if not (ROOT / 'ml').exists():
    print('Ejecuta este notebook desde la raíz del repositorio o sube la carpeta ml.')
sys.path.insert(0, str(ROOT))

## 1. Pandas, NumPy y dataset de demostración

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ml.generate_demo_dataset import generate_rows

DATA_SOURCE = 'synthetic-demo'  # Cambia por la ruta de un CSV anonimizado real.
df = pd.DataFrame(generate_rows(driver_count=18, windows_per_driver=40, seed=42))
display(df.head())
print('Fuente:', df.dataSource.unique(), '| Filas:', len(df))

## 2. Estadística básica y probabilidades

In [ ]:
stats = pd.DataFrame({
    'media': df[['fatiguePct','avgEAR','perclos']].mean(),
    'mediana': df[['fatiguePct','avgEAR','perclos']].median(),
    'varianza_muestral': df[['fatiguePct','avgEAR','perclos']].var(ddof=1),
    'desviacion_estandar': df[['fatiguePct','avgEAR','perclos']].std(ddof=1),
})
display(stats)
print('P(fatiga ≥ 60):', (df.fatiguePct >= 60).mean())
print('P(microsueño):', (df.microsleeps > 0).mean())
sns.heatmap(df.select_dtypes('number').corr(), cmap='coolwarm', center=0)
plt.title('Correlaciones de características'); plt.show()

## 3. Regresión lineal simple, múltiple y álgebra lineal

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_simple = df[['elapsedMinutes']].to_numpy()  # matriz n×1
y = df['fatiguePct'].to_numpy()              # vector n
simple = LinearRegression().fit(X_simple, y)
pred_simple = simple.predict(X_simple)
print('Simple: pendiente=', simple.coef_[0], 'intercepto=', simple.intercept_, 'R²=', r2_score(y, pred_simple))
sns.scatterplot(data=df.sample(180, random_state=42), x='elapsedMinutes', y='fatiguePct', alpha=.45)
order = np.argsort(X_simple[:,0]); plt.plot(X_simple[order,0], pred_simple[order], color='crimson'); plt.show()

features = ['elapsedMinutes','avgEAR','minEAR','perclos','blinkRate','microsleeps','facePresenceRatio']
X = df[features].to_numpy()
X_design = np.column_stack([np.ones(len(X)), X])
beta = np.linalg.pinv(X_design.T @ X_design) @ X_design.T @ y
y_hat = X_design @ beta
print('X:', X_design.shape, 'β:', beta.shape, 'ŷ:', y_hat.shape)
print('β =', beta)
print({'R2': r2_score(y,y_hat), 'MAE': mean_absolute_error(y,y_hat), 'RMSE': mean_squared_error(y,y_hat)**0.5})

## 4. Segmentación no supervisada: K-Means, codo y Silhouette

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
cluster_features = ['fatiguePct','avgEAR','perclos','blinkRate','microsleeps']
Z = StandardScaler().fit_transform(df[cluster_features])
scores, inertias = {}, {}
for k in range(2, 7):
    candidate = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
    inertias[k] = candidate.inertia_
    scores[k] = silhouette_score(Z, candidate.labels_)
best_k = max(scores, key=scores.get)
model_kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=42).fit(Z)
df['cluster'] = model_kmeans.labels_
fig, ax = plt.subplots(1,2,figsize=(10,4))
ax[0].plot(list(inertias), list(inertias.values()), marker='o'); ax[0].set_title('Método del codo')
ax[1].plot(list(scores), list(scores.values()), marker='o'); ax[1].set_title('Silhouette')
plt.show(); display(df.groupby('cluster')[cluster_features].mean())

## 5. OpenCV sobre un video autorizado (opcional)

In [ ]:
from ml.preprocess_video import analyze_video
VIDEO_AUTORIZADO = None  # Ejemplo: Path('/content/video-autorizado.mp4')
if VIDEO_AUTORIZADO:
    metrics = analyze_video(Path(VIDEO_AUTORIZADO), Path('/content/opencv-metrics.csv'))
    display(pd.DataFrame(metrics).head())
else:
    print('Celda omitida: no se proporcionó un video autorizado.')

## 6. Red neuronal personalizada TensorFlow/Keras

In [ ]:
import tensorflow as tf
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score
classes = ['normal','atencion','peligro']
encoder = LabelEncoder().fit(classes)
labels = encoder.transform(df.targetClass)
groups = df.driverId.astype(str).to_numpy()
split = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=42)
train_idx, test_idx = next(split.split(X, labels, groups=groups))
scaler = StandardScaler().fit(X[train_idx])
x_train, x_test = scaler.transform(X[train_idx]), scaler.transform(X[test_idx])
y_train, y_test = labels[train_idx], labels[test_idx]
network = tf.keras.Sequential([
    tf.keras.layers.Input((len(features),)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(.20),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax'),
])
network.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = network.fit(x_train, y_train, validation_split=.2, epochs=100, batch_size=32,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True)], verbose=0)
pred = network.predict(x_test, verbose=0).argmax(axis=1)
print(classification_report(y_test, pred, target_names=classes, zero_division=0))
matrix = confusion_matrix(y_test, pred, labels=range(3))
sns.heatmap(matrix, annot=True, fmt='d', xticklabels=classes, yticklabels=classes); plt.show()
pd.DataFrame(history.history).plot(subplots=True, figsize=(8,6)); plt.show()

## 7. Exportación segura para inferencia web

In [ ]:
from ml.export_model_json import export_keras_model
output = Path('/content/somnoguard-company-model.json')
artifact = export_keras_model(network, scaler, output, {
    'trainingSource': DATA_SOURCE,
    'validated': False,
    'warning': 'Modelo académico; requiere validación independiente.'
})
print('Artefacto:', output, '| validated:', artifact['validated'])
network.save('/content/somnoguard-company-model.keras')